In [0]:

# ============================================================
# 09_EVALUATION
# CELL 1: DISCOVER EXISTING BRONZE TABLES
# ============================================================

from pyspark.sql import functions as F

BRONZE_SCHEMA = "workspace.firmable_v2_bronze"
SILVER_SCHEMA = "workspace.firmable_v2_silver"
EVALUATION_SCHEMA = "workspace.firmable_v2_evaluation"

# List all Bronze tables
bronze_tables = spark.sql(
    f"SHOW TABLES IN {BRONZE_SCHEMA}"
)

print("=" * 65)
print("BRONZE TABLES")
print("=" * 65)

bronze_tables.show(100, truncate=False)

# List Silver tables
silver_tables = spark.sql(
    f"SHOW TABLES IN {SILVER_SCHEMA}"
)

print("\n" + "=" * 65)
print("SILVER TABLES")
print("=" * 65)

silver_tables.show(100, truncate=False)

# List existing Evaluation tables
evaluation_tables = spark.sql(
    f"SHOW TABLES IN {EVALUATION_SCHEMA}"
)

print("\n" + "=" * 65)
print("EVALUATION TABLES")
print("=" * 65)

evaluation_tables.show(100, truncate=False)

print("\nTABLE DISCOVERY COMPLETED")


In [0]:

# ============================================================
# CELL 2: INSPECT GROUND-TRUTH DATA
# ============================================================

from pyspark.sql import functions as F

GROUND_TRUTH_TABLE = "workspace.firmable_v2_bronze.ground_truth"

ground_truth = spark.table(GROUND_TRUTH_TABLE)

print("=" * 65)
print("GROUND-TRUTH TABLE SCHEMA")
print("=" * 65)

ground_truth.printSchema()

print("\nGROUND-TRUTH COLUMN NAMES")
print(ground_truth.columns)

print("\nGROUND-TRUTH RECORD COUNT")
print(ground_truth.count())

print("\nGROUND-TRUTH SAMPLE RECORDS")
ground_truth.show(10, truncate=False)

print("\nGROUND-TRUTH NULL COUNTS")

ground_truth.select([
    F.sum(
        F.col(column).isNull().cast("int")
    ).alias(column)
    for column in ground_truth.columns
]).show(truncate=False)

print("\nGROUND-TRUTH INSPECTION COMPLETED")


In [0]:

# ============================================================
# CELL 3: COMPARE PREDICTED AND ACTUAL MANAGER RELATIONSHIPS
#
# Ground truth is used exclusively for evaluation.
# ============================================================

from pyspark.sql import functions as F

# ------------------------------------------------------------
# 1. PREPARE GROUND TRUTH
# ------------------------------------------------------------

actual_relationships = ground_truth.select(
    "company_id",
    "employee_id",
    F.col("actual_manager_id").alias("actual_manager_id")
)

# ------------------------------------------------------------
# 2. PREPARE PREDICTED RELATIONSHIPS
# ------------------------------------------------------------

predicted_relationships = assignments.select(
    "company_id",
    "employee_id",
    F.col("candidate_manager_id").alias("predicted_manager_id"),
    "assignment_status",
    "candidate_score"
)

# ------------------------------------------------------------
# 3. JOIN ACTUAL AND PREDICTED RELATIONSHIPS
# ------------------------------------------------------------

evaluation_comparison = (
    actual_relationships
    .join(
        predicted_relationships,
        ["company_id", "employee_id"],
        "inner"
    )
)

# ------------------------------------------------------------
# 4. CLASSIFY EACH EMPLOYEE'S PREDICTION
#
# TP: Predicted manager matches actual manager.
# FP: A manager was predicted, but it is incorrect.
# FN: Actual manager exists, but correct manager was not predicted.
# CORRECT_ROOT: Both actual and predicted manager are NULL.
# ABSTAINED: Actual manager exists, but no manager was predicted.
# ------------------------------------------------------------

evaluation_comparison = evaluation_comparison.withColumn(
    "evaluation_status",
    F.when(
        F.col("actual_manager_id").isNull() &
        F.col("predicted_manager_id").isNull(),
        "CORRECT_ROOT"
    )
    .when(
        F.col("actual_manager_id").isNotNull() &
        (F.col("actual_manager_id") == F.col("predicted_manager_id")),
        "TP"
    )
    .when(
        F.col("actual_manager_id").isNotNull() &
        F.col("predicted_manager_id").isNotNull() &
        (F.col("actual_manager_id") != F.col("predicted_manager_id")),
        "FP_FN"
    )
    .when(
        F.col("actual_manager_id").isNotNull() &
        F.col("predicted_manager_id").isNull(),
        "FN"
    )
    .when(
        F.col("actual_manager_id").isNull() &
        F.col("predicted_manager_id").isNotNull(),
        "FP"
    )
    .otherwise("UNCLASSIFIED")
)

# ------------------------------------------------------------
# 5. CALCULATE OVERALL COUNTS
# ------------------------------------------------------------

total_employees = evaluation_comparison.count()

actual_relationship_count = evaluation_comparison.filter(
    F.col("actual_manager_id").isNotNull()
).count()

predicted_relationship_count = evaluation_comparison.filter(
    F.col("predicted_manager_id").isNotNull()
).count()

true_positives = evaluation_comparison.filter(
    F.col("evaluation_status") == "TP"
).count()

false_positives = evaluation_comparison.filter(
    F.col("evaluation_status").isin("FP", "FP_FN")
).count()

false_negatives = evaluation_comparison.filter(
    F.col("evaluation_status").isin("FN", "FP_FN")
).count()

correct_roots = evaluation_comparison.filter(
    F.col("evaluation_status") == "CORRECT_ROOT"
).count()

abstained_employees = evaluation_comparison.filter(
    F.col("evaluation_status") == "FN"
).count()

# ------------------------------------------------------------
# 6. CALCULATE METRICS
# ------------------------------------------------------------

precision = (
    true_positives / predicted_relationship_count
    if predicted_relationship_count > 0 else 0
)

recall = (
    true_positives / actual_relationship_count
    if actual_relationship_count > 0 else 0
)

f1_score = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) > 0 else 0
)

coverage = (
    predicted_relationship_count / actual_relationship_count
    if actual_relationship_count > 0 else 0
)

# ------------------------------------------------------------
# 7. DISPLAY RESULTS
# ------------------------------------------------------------

print("=" * 70)
print("FIRMABLE - MANAGER RELATIONSHIP EVALUATION")
print("=" * 70)

print("\nDATASET SUMMARY")

print("Total employees:", total_employees)
print("Actual reporting relationships:", actual_relationship_count)
print("Predicted reporting relationships:", predicted_relationship_count)
print("Correctly identified roots:", correct_roots)

print("\nEVALUATION COUNTS")

print("True positives:", true_positives)
print("False positives:", false_positives)
print("False negatives:", false_negatives)
print("Abstained employees with actual managers:", abstained_employees)

print("\nEVALUATION METRICS")

print(f"Precision: {precision * 100:.2f}%")
print(f"Recall: {recall * 100:.2f}%")
print(f"F1 Score: {f1_score * 100:.2f}%")
print(f"Assignment Coverage: {coverage * 100:.2f}%")

print("\nEVALUATION STATUS DISTRIBUTION")

evaluation_comparison.groupBy(
    "evaluation_status"
).count().orderBy(
    "evaluation_status"
).show(truncate=False)

# ------------------------------------------------------------
# 8. BASIC INTEGRITY CHECKS
# ------------------------------------------------------------

assert total_employees == 89576, (
    f"Unexpected evaluation row count: {total_employees}"
)

assert actual_relationship_count == 89076, (
    "Actual relationship count does not match expected count."
)

assert predicted_relationship_count == 86223, (
    "Predicted relationship count does not match assignment output."
)

assert true_positives + false_negatives == actual_relationship_count

assert true_positives + false_positives == predicted_relationship_count

assert evaluation_comparison.filter(
    F.col("evaluation_status") == "UNCLASSIFIED"
).count() == 0

print("\nCELL 3 EVALUATION COMPLETED SUCCESSFULLY")


In [0]:

# ============================================================
# CELL 4: EVALUATION BY COMPANY STRUCTURE
# ============================================================

# Load company metadata
companies = spark.table(
    "workspace.firmable_v2_bronze.companies"
).select(
    "company_id",
    "org_structure"
).dropDuplicates(["company_id"])

# Join evaluation results with company structure
structure_evaluation = evaluation_comparison.join(
    companies,
    "company_id",
    "inner"
)

# Aggregate metrics by organizational structure
company_structure_metrics = (
    structure_evaluation
    .groupBy("org_structure")
    .agg(
        F.count("*").alias("total_employees"),

        F.sum(
            F.when(
                F.col("actual_manager_id").isNotNull(), 1
            ).otherwise(0)
        ).alias("actual_relationships"),

        F.sum(
            F.when(
                F.col("predicted_manager_id").isNotNull(), 1
            ).otherwise(0)
        ).alias("predicted_relationships"),

        F.sum(
            F.when(
                F.col("evaluation_status") == "TP", 1
            ).otherwise(0)
        ).alias("true_positives"),

        F.sum(
            F.when(
                F.col("evaluation_status").isin("FP", "FP_FN"), 1
            ).otherwise(0)
        ).alias("false_positives"),

        F.sum(
            F.when(
                F.col("evaluation_status").isin("FN", "FP_FN"), 1
            ).otherwise(0)
        ).alias("false_negatives"),

        F.sum(
            F.when(
                F.col("predicted_manager_id").isNotNull(), 1
            ).otherwise(0)
        ).alias("assigned_employees")
    )
)

# Calculate precision, recall, F1 and employee assignment coverage
company_structure_metrics = (
    company_structure_metrics
    .withColumn(
        "precision",
        F.when(
            F.col("predicted_relationships") > 0,
            F.col("true_positives") /
            F.col("predicted_relationships")
        ).otherwise(0)
    )
    .withColumn(
        "recall",
        F.when(
            F.col("actual_relationships") > 0,
            F.col("true_positives") /
            F.col("actual_relationships")
        ).otherwise(0)
    )
    .withColumn(
        "f1_score",
        F.when(
            (F.col("precision") + F.col("recall")) > 0,
            2 * F.col("precision") * F.col("recall") /
            (F.col("precision") + F.col("recall"))
        ).otherwise(0)
    )
    .withColumn(
        "assignment_coverage",
        F.col("assigned_employees") /
        F.col("total_employees")
    )
)

# Display percentages
print("=" * 70)
print("EVALUATION BY COMPANY ORGANIZATIONAL STRUCTURE")
print("=" * 70)

company_structure_metrics.select(
    "org_structure",
    "total_employees",
    "true_positives",
    "false_positives",
    "false_negatives",
    F.round(F.col("precision") * 100, 2).alias("precision_pct"),
    F.round(F.col("recall") * 100, 2).alias("recall_pct"),
    F.round(F.col("f1_score") * 100, 2).alias("f1_pct"),
    F.round(
        F.col("assignment_coverage") * 100, 2
    ).alias("coverage_pct")
).orderBy(
    "org_structure"
).show(truncate=False)

print("\nCELL 4 COMPLETED SUCCESSFULLY")


In [0]:
from pyspark.sql import functions as F

ground_truth = spark.table("workspace.firmable_v2_bronze.ground_truth")

assignments = spark.table(
    "workspace.firmable_v2_silver.manager_assignments"
)

companies = spark.table("workspace.firmable_v2_bronze.companies")

# Keep only actual manager assignments
predicted = (
    assignments
    .filter(F.col("assignment_status") == "ASSIGNED")
    .select(
        "company_id",
        "employee_id",
        F.col("candidate_manager_id").alias("predicted_manager_id")
    )
)

# Compare predictions against isolated ground truth
comparison = (
    ground_truth.alias("g")
    .join(predicted.alias("p"), ["company_id", "employee_id"], "left")
    .withColumn(
        "evaluation_status",
        F.when(
            F.col("g.actual_manager_id").isNull() &
            F.col("p.predicted_manager_id").isNull(), "CORRECT_ROOT"
        )
        .when(
            F.col("g.actual_manager_id") ==
            F.col("p.predicted_manager_id"), "TP"
        )
        .when(
            F.col("g.actual_manager_id").isNotNull() &
            F.col("p.predicted_manager_id").isNull(), "FN"
        )
        .when(
            F.col("g.actual_manager_id").isNull() &
            F.col("p.predicted_manager_id").isNotNull(), "FP"
        )
        .otherwise("FP_FN")
    )
)

# Aggregate evaluation metrics by organizational structure
structure_metrics = (
    comparison
    .join(
        companies.select("company_id", "org_structure"),
        "company_id"
    )
    .groupBy("org_structure")
    .agg(
        F.count("*").alias("total_employees"),

        F.sum(
            F.when(F.col("evaluation_status") == "TP", 1).otherwise(0)
        ).alias("true_positives"),

        F.sum(
            F.when(
                F.col("evaluation_status").isin("FP", "FP_FN"), 1
            ).otherwise(0)
        ).alias("false_positives"),

        F.sum(
            F.when(
                F.col("evaluation_status").isin("FN", "FP_FN"), 1
            ).otherwise(0)
        ).alias("false_negatives"),

        F.sum(
            F.when(F.col("predicted_manager_id").isNotNull(), 1).otherwise(0)
        ).alias("assigned_employees")
    )
    .withColumn(
        "precision",
        F.try_divide(
            F.col("true_positives"),
            F.col("true_positives") + F.col("false_positives")
        )
    )
    .withColumn(
        "recall",
        F.try_divide(
            F.col("true_positives"),
            F.col("true_positives") + F.col("false_negatives")
        )
    )
    .withColumn(
        "f1_score",
        F.try_divide(
            2 * F.col("precision") * F.col("recall"),
            F.col("precision") + F.col("recall")
        )
    )
    .withColumn(
        "assignment_coverage",
        F.try_divide(
            F.col("assigned_employees"),
            F.col("total_employees")
        )
    )
)

# Persist evaluation metrics
(
    structure_metrics
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.firmable_v2_evaluation.company_structure_metrics"
    )
)

print("Company structure evaluation metrics saved successfully.")

display(
    structure_metrics
    .orderBy("org_structure")
)

In [0]:
from pyspark.sql import functions as F

ground_truth = spark.table(
    "workspace.firmable_v2_bronze.ground_truth"
)

assignments = spark.table(
    "workspace.firmable_v2_silver.manager_assignments"
)

predicted = (
    assignments
    .filter(F.col("assignment_status") == "ASSIGNED")
    .select(
        "company_id",
        "employee_id",
        F.col("candidate_manager_id").alias("predicted_manager_id"),
        "candidate_score",
        "relationship_type",
        "assignment_round"
    )
)

error_analysis = (
    ground_truth.alias("g")
    .join(predicted.alias("p"), ["company_id", "employee_id"], "left")
    .withColumn(
        "evaluation_status",
        F.when(
            F.col("g.actual_manager_id").isNull() &
            F.col("p.predicted_manager_id").isNull(), "CORRECT_ROOT"
        )
        .when(
            F.col("g.actual_manager_id") ==
            F.col("p.predicted_manager_id"), "TP"
        )
        .when(
            F.col("g.actual_manager_id").isNotNull() &
            F.col("p.predicted_manager_id").isNull(), "FN"
        )
        .when(
            F.col("g.actual_manager_id").isNull() &
            F.col("p.predicted_manager_id").isNotNull(), "FP"
        )
        .otherwise("FP_FN")
    )
    .select(
        "company_id",
        "employee_id",
        F.col("g.actual_manager_id").alias("actual_manager_id"),
        "predicted_manager_id",
        "candidate_score",
        "relationship_type",
        "assignment_round",
        "evaluation_status"
    )
)

(
    error_analysis.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.firmable_v2_evaluation.error_analysis_employee_level"
    )
)

print("Employee-level error analysis saved successfully.")

display(
    error_analysis
    .groupBy("evaluation_status")
    .count()
    .orderBy("evaluation_status")
)

In [0]:
from pyspark.sql import functions as F

error_analysis = spark.table(
    "workspace.firmable_v2_evaluation.error_analysis_employee_level"
)

assignments = spark.table(
    "workspace.firmable_v2_silver.manager_assignments"
)

companies = spark.table(
    "workspace.firmable_v2_bronze.companies"
)

# Rename diagnostic columns to avoid conflicts
assignment_diagnostics = assignments.select(
    "company_id",
    "employee_id",
    F.col("assignment_status").alias("diagnostic_assignment_status"),
    "candidate_count",
    "best_candidate_score",
    "score_margin",
    F.col("relationship_type").alias("diagnostic_relationship_type"),
    F.col("assignment_round").alias("diagnostic_assignment_round")
)

root_cause_analysis = (
    error_analysis
    .join(
        assignment_diagnostics,
        ["company_id", "employee_id"],
        "left"
    )
    .join(
        companies.select("company_id", "org_structure"),
        "company_id",
        "left"
    )
    .withColumn(
        "root_cause",
        F.when(
            F.col("evaluation_status") == "FP_FN",
            "WRONG_MANAGER_SELECTED"
        )
        .when(
            (F.col("evaluation_status") == "FN") &
            (F.col("diagnostic_assignment_status") == "ABSTAINED_LOW_SCORE"),
            "LOW_CANDIDATE_SCORE"
        )
        .when(
            (F.col("evaluation_status") == "FN") &
            (F.col("diagnostic_assignment_status") == "NO_CANDIDATE_OR_ROOT"),
            "NO_ELIGIBLE_MANAGER_CANDIDATE"
        )
        .when(
            (F.col("evaluation_status") == "FN") &
            (F.col("diagnostic_assignment_status") == "UNASSIGNED_CAPACITY_OR_ELIGIBILITY"),
            "CAPACITY_OR_ELIGIBILITY_CONSTRAINT"
        )
        .when(
            F.col("evaluation_status") == "FN",
            "OTHER_UNASSIGNED_CASE"
        )
        .when(
            F.col("evaluation_status") == "TP",
            "CORRECT_MANAGER_SELECTED"
        )
        .otherwise("CORRECT_ROOT")
    )
)

root_cause_summary = (
    root_cause_analysis
    .groupBy(
        "root_cause",
        "evaluation_status",
        "diagnostic_assignment_status",
        "org_structure",
        "diagnostic_relationship_type"
    )
    .agg(
        F.count("*").alias("employee_count"),
        F.avg("candidate_count").alias("avg_candidate_count"),
        F.avg("best_candidate_score").alias("avg_best_candidate_score"),
        F.avg("score_margin").alias("avg_score_margin")
    )
    .orderBy(F.desc("employee_count"))
)

(
    root_cause_summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.firmable_v2_evaluation.error_analysis_root_causes"
    )
)

print("Root-cause analysis saved successfully.")
display(root_cause_summary)

In [0]:
from pyspark.sql import functions as F

candidate_scores = spark.table(
    "workspace.firmable_v2_silver.manager_candidate_scores"
)

problematic_candidates = (
    candidate_scores
    .filter(
        F.col("relationship_type") ==
        "Team Manager to Individual Contributor"
    )
)

print("PROBLEMATIC RELATIONSHIP RULE")
print("Total candidate relationships:", problematic_candidates.count())

display(
    problematic_candidates
    .groupBy("candidate_score", "candidate_priority")
    .count()
    .orderBy("candidate_score", "candidate_priority")
)

In [0]:
from pyspark.sql import functions as F

errors = spark.table(
    "workspace.firmable_v2_evaluation.error_analysis_employee_level"
).filter(
    F.col("evaluation_status") == "FP_FN"
)

employees = spark.table(
    "workspace.firmable_v2_silver.employees_clean"
)

wrong_assignments = (
    errors
    .join(
        employees.select(
            "company_id",
            F.col("employee_id").alias("employee_id"),
            F.col("job_title_clean").alias("employee_title"),
            F.col("department").alias("employee_department"),
            F.col("business_unit").alias("employee_business_unit"),
            F.col("team_id").alias("employee_team_id")
        ),
        ["company_id", "employee_id"]
    )
    .join(
        employees.select(
            "company_id",
            F.col("employee_id").alias("predicted_manager_id"),
            F.col("job_title_clean").alias("manager_title"),
            F.col("department").alias("manager_department"),
            F.col("business_unit").alias("manager_business_unit"),
            F.col("team_id").alias("manager_team_id")
        ),
        ["company_id", "predicted_manager_id"]
    )
    .withColumn(
        "same_team",
        F.col("employee_team_id") == F.col("manager_team_id")
    )
    .withColumn(
        "same_department",
        F.col("employee_department") == F.col("manager_department")
    )
    .withColumn(
        "same_business_unit",
        F.col("employee_business_unit") == F.col("manager_business_unit")
    )
)

display(
    wrong_assignments
    .groupBy(
        "employee_title",
        "manager_title",
        "same_team",
        "same_department",
        "same_business_unit"
    )
    .count()
    .orderBy(F.desc("count"))
)